# Digital Burnout Detection
Run cells top to bottom. Files stay on disk — no re-uploading ever.

In [14]:
# Cell 1 — Install dependencies (run once)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'torch', 'torchvision', 'torchaudio',
    'scikit-learn', 'pandas', 'numpy',
    'joblib', 'gradio', '--quiet'])

CompletedProcess(args=['/Users/pragatik/digital_burnout/venv/bin/python', '-m', 'pip', 'install', 'torch', 'torchvision', 'torchaudio', 'scikit-learn', 'pandas', 'numpy', 'joblib', 'gradio', '--quiet'], returncode=0)

In [16]:
# Cell 2 — Imports & paths
import os
os.chdir(os.path.expanduser('~/digital_burnout'))
print(os.getcwd())  # should print /Users/pragatik/digital_burnout
import json, warnings, pickle, os
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

# ── paths (all relative — run notebook from digital_burnout/ folder)
ROOT      = Path('student life')
SENSING   = ROOT / 'sensing'
APP_USAGE = ROOT / 'app_usage'
CALL_LOG  = ROOT / 'call_log'
SMS_DIR   = ROOT / 'sms'
DINING    = ROOT / 'dinning'
SURVEY    = ROOT / 'survey'
SCREENTIME_CSV = 'ScreenTime vs MentalWellness.csv'
CACHE_FILE     = 'sensing_cache.pkl'
MODEL_PATH     = 'models/best_burnout_model.pt'
SCALER_PATH    = 'models/burnout_scaler.pkl'
FEATS_PATH     = 'models/feature_cols.pkl'
DATA_PATH      = 'models/unified_burnout_dataset.csv'

os.makedirs('models', exist_ok=True)

# ── device (MPS = Apple M1 GPU)
if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'
print(f'Device : {DEVICE}')

# ── verify folders
for name, p in [('sensing',SENSING),('app_usage',APP_USAGE),
                ('call_log',CALL_LOG),('sms',SMS_DIR),
                ('dinning',DINING),('survey',SURVEY)]:
    print(f"{'OK' if p.exists() else 'MISSING'}  {name}")

/Users/pragatik/digital_burnout
Device : mps
OK  sensing
OK  app_usage
OK  call_log
OK  sms
OK  dinning
OK  survey


In [17]:
# Cell 3 — Helper functions
def _uid(stem):
    for p in stem.split('_'):
        if p.startswith('u') and p[1:].isdigit():
            return p
    return stem

def _to_date(series):
    return pd.to_datetime(pd.to_numeric(series, errors='coerce'), unit='s').dt.date

print('Helpers loaded')

Helpers loaded


In [18]:
# Cell 4 — Load burnout labels from PSS survey
PSS_SCALE   = {'Never':0,'Almost never':1,'Sometime':2,'Fairly often':3,'Very often':4}
PSS_REVERSE = [3,4,6,7]
PSQI_QUALITY = {'Very good':5,'Fairly good':4,'Fairly bad':2,'Very bad':1}

def load_burnout_labels():
    pss = pd.read_csv(SURVEY/'PerceivedStressScale.csv')
    pss.columns = [c.strip() for c in pss.columns]
    q_cols = [c for c in pss.columns if c not in ['uid','type']]
    rows = []
    for _, row in pss.iterrows():
        scores = []
        for i, col in enumerate(q_cols):
            val = PSS_SCALE.get(str(row[col]).strip(), np.nan)
            if np.isnan(val): continue
            if i in PSS_REVERSE: val = 4 - val
            scores.append(val)
        if scores:
            rows.append({'uid': row['uid'], 'burnout_score': sum(scores)/40*9+1})
    df = pd.DataFrame(rows).groupby('uid')['burnout_score'].mean().reset_index()
    print(f'  Burnout labels : {len(df)} students | {df.burnout_score.min():.2f}–{df.burnout_score.max():.2f}')
    return df

def load_sleep_labels():
    psqi = pd.read_csv(SURVEY/'psqi.csv')
    psqi.columns = [c.strip() for c in psqi.columns]
    hours_col   = [c for c in psqi.columns if 'hours of actual sleep' in c.lower()][0]
    quality_col = [c for c in psqi.columns if 'rate your sleep quality' in c.lower()][0]
    psqi['sleep_hours']   = pd.to_numeric(psqi[hours_col], errors='coerce')
    psqi['sleep_quality'] = psqi[quality_col].map(PSQI_QUALITY)
    out = psqi.groupby('uid')[['sleep_hours','sleep_quality']].mean().reset_index()
    print(f'  Sleep labels   : {len(out)} students')
    return out

burnout = load_burnout_labels()
sleep   = load_sleep_labels()

  Burnout labels : 46 students | 2.24–8.54
  Sleep labels   : 46 students


In [19]:
# Cell 5 — Sensing feature extraction (cached)
# Processes one student at a time — no memory crashes
# Re-running this cell loads from cache if it already exists

if Path(CACHE_FILE).exists():
    with open(CACHE_FILE, 'rb') as f:
        agg_sensing = pickle.load(f)
    print(f'Loaded from cache — shape: {agg_sensing.shape}')
else:
    all_uids = burnout['uid'].tolist()
    print(f'Extracting features for {len(all_uids)} students...')
    results = []

    social_kw     = {'instagram','facebook','twitter','snapchat','tiktok',
                     'whatsapp','messenger','telegram','reddit','discord'}
    productive_kw = {'docs','sheets','calendar','office','word','excel',
                     'notion','evernote','classroom','canvas'}

    for uid in all_uids:
        row = {'uid': uid}

        # screen
        try:
            f = SENSING/'phonelock'/f'{uid}_phonelock.csv'
            if f.exists():
                df = pd.read_csv(f, header=None, names=['ts','st'])
                df['ts'] = pd.to_numeric(df['ts'], errors='coerce')
                df['st'] = pd.to_numeric(df['st'], errors='coerce')
                df = df.dropna().sort_values('ts')
                ts, st = df['ts'].values, df['st'].values
                hrs_arr = pd.to_datetime(ts, unit='s').hour
                sessions = []
                for i,(t,s) in enumerate(zip(ts,st)):
                    if s==1:
                        nxt = ts[(i+1):][st[(i+1):]==0]
                        if len(nxt):
                            dur=(nxt[0]-t)/60.0
                            if 0<dur<120: sessions.append((dur,hrs_arr[i]))
                total = sum(d for d,_ in sessions)
                eve   = sum(d for d,h in sessions if h>=21)
                row['screen_time_hours']    = round(total/60,3)
                row['unlock_count']         = len(sessions)
                row['avg_session_min']      = round(total/len(sessions),2) if sessions else 0.0
                row['evening_screen_ratio'] = round(eve/total,3) if total>0 else 0.0
        except: pass

        # activity
        try:
            f = SENSING/'activity'/f'{uid}_activity_inference.csv'
            if f.exists():
                df = pd.read_csv(f, header=None, names=['ts','act'])
                df['act'] = pd.to_numeric(df['act'], errors='coerce').dropna()
                total = max(len(df),1)
                row['stationary_ratio'] = (df['act']==0).sum()/total
                row['active_ratio']     = ((df['act']==1)|(df['act']==2)).sum()/total
        except: pass

        # audio
        try:
            f = SENSING/'audio'/f'{uid}_audio_inference.csv'
            if f.exists():
                df = pd.read_csv(f, header=None, names=['ts','aud'])
                df['aud'] = pd.to_numeric(df['aud'], errors='coerce')
                total = max(len(df),1)
                row['social_audio_ratio'] = (df['aud']==1).sum()/total
        except: pass

        # conversation
        try:
            f = SENSING/'conversation'/f'{uid}_conversation.csv'
            if f.exists():
                df = pd.read_csv(f, header=None, names=['start','end','inf'])
                df['start'] = pd.to_numeric(df['start'], errors='coerce')
                df['end']   = pd.to_numeric(df['end'],   errors='coerce')
                df = df.dropna(subset=['start','end'])
                df['dur'] = ((df['end']-df['start'])/60.0).clip(lower=0)
                row['conversation_count'] = len(df)
                row['total_conv_min']     = df['dur'].sum()
        except: pass

        # apps
        try:
            f = APP_USAGE/f'running_app_{uid}.csv'
            if f.exists():
                df = pd.read_csv(f, header=None, names=['ts','apps'])
                df['apps'] = df['apps'].fillna('').astype(str).str.lower()
                all_apps = df['apps'].str.split('|').explode().str.strip()
                all_apps = all_apps[all_apps!='']
                total    = max(len(all_apps),1)
                top_app  = df['apps'].str.split('|').str[0].str.strip()
                switches = (top_app!=top_app.shift()).sum()
                row['app_switches_per_hour'] = round(switches/max(len(df)/60.0,0.1),2)
                row['social_app_ratio']      = round(all_apps.apply(lambda a:any(k in a for k in social_kw)).sum()/total,3)
                row['productivity_ratio']    = round(all_apps.apply(lambda a:any(k in a for k in productive_kw)).sum()/total,3)
                row['unique_apps_per_day']   = all_apps.nunique()
        except: pass

        # calls
        try:
            f = CALL_LOG/f'call_log_{uid}.csv'
            if f.exists():
                df = pd.read_csv(f, header=None, names=['ts','dur','ctype'])
                df['dur']   = pd.to_numeric(df['dur'],   errors='coerce').fillna(0)
                df['ctype'] = pd.to_numeric(df['ctype'], errors='coerce')
                row['call_count']        = len(df)
                row['total_call_min']    = df['dur'].sum()/60.0
                row['missed_call_ratio'] = (df['ctype']==3).sum()/max(len(df),1)
        except: pass

        # sms
        try:
            f = SMS_DIR/f'sms_{uid}.csv'
            if f.exists():
                df = pd.read_csv(f, header=None, names=['ts','stype'])
                df['stype'] = pd.to_numeric(df['stype'], errors='coerce')
                row['sms_count']      = len(df)
                row['sms_sent_ratio'] = (df['stype']==2).sum()/max(len(df),1)
        except: pass

        # dining
        try:
            f = DINING/f'dinning_{uid}.csv'
            if f.exists():
                df = pd.read_csv(f, header=None, names=['ts','did'])
                row['dining_visits'] = len(df)
        except: pass

        results.append(row)
        print(f'  {uid}', end='  ')

    agg_sensing = pd.DataFrame(results)
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(agg_sensing, f)
    print(f'\nShape: {agg_sensing.shape} — cached to {CACHE_FILE}')

Extracting features for 46 students...
  u00    u01    u02    u03    u04    u05    u07    u08    u09    u10    u12    u13    u14    u15    u16    u17    u18    u19    u20    u22    u23    u24    u27    u30    u31    u32    u33    u34    u35    u36    u42    u43    u44    u45    u46    u47    u49    u50    u51    u52    u53    u54    u56    u57    u58    u59  
Shape: (46, 10) — cached to sensing_cache.pkl


In [20]:
# Cell 6 — Merge StudentLife + ScreenTime survey

# StudentLife
sl = burnout.merge(sleep, on='uid', how='left')
sl = sl.merge(agg_sensing, on='uid', how='left')
sl = sl.rename(columns={'burnout_score': 'burnout_score'})
sl = sl.fillna(sl.median(numeric_only=True))
print(f'StudentLife : {len(sl)} rows | burnout mean {sl.burnout_score.mean():.2f}')

# ScreenTime survey — fix skew by clipping to SL range
sv = pd.read_csv(SCREENTIME_CSV)
sv.columns = [c.strip() for c in sv.columns]
sv = sv.loc[:, ~sv.columns.str.startswith('Unnamed')]
sv = sv.rename(columns={
    'stress_level_0_10':          'burnout_score',
    'sleep_hours':                'sleep_hours',
    'sleep_quality_1_5':          'sleep_quality',
    'screen_time_hours':          'screen_time_hours',
    'exercise_minutes_per_week':  'exercise_min_per_week',
    'social_hours_per_week':      'social_hours_per_week',
    'productivity_0_100':         'productivity_ratio',
})
sv['burnout_score'] = pd.to_numeric(sv['burnout_score'], errors='coerce')
sv = sv.dropna(subset=['burnout_score'])

# clip to StudentLife distribution range
sl_min, sl_max = sl['burnout_score'].min(), sl['burnout_score'].max()
sv['burnout_score'] = sv['burnout_score'].clip(sl_min, sl_max)

# balance survey — undersample high burnout rows
low  = sv[sv['burnout_score'] < 4]
mid  = sv[(sv['burnout_score'] >= 4) & (sv['burnout_score'] < 7)]
high = sv[sv['burnout_score'] >= 7]
target = max(len(low), len(mid), 30)
high_bal = high.sample(min(len(high), target * 2), random_state=42)
sv = pd.concat([low, mid, high_bal], ignore_index=True)
sv['uid'] = [f'sv_{i:04d}' for i in range(len(sv))]
print(f'Survey      : {len(sv)} rows after balancing | low={len(low)} mid={len(mid)} high={len(high_bal)}')

# combine
unified = pd.concat([sl, sv], ignore_index=True)
unified = unified.fillna(unified.median(numeric_only=True))

# drop leakage columns
drop_cols = {'age','work_screen_hours','leisure_screen_hours',
             'mental_wellness_index_0_100','gender','occupation','work_mode'}
unified = unified.drop(columns=[c for c in drop_cols if c in unified.columns])

# feature columns
exclude   = {'burnout_score','uid'}
feat_cols = [c for c in unified.select_dtypes(include=[np.number]).columns
             if c not in exclude]

# scale
scaler = StandardScaler()
unified[feat_cols] = scaler.fit_transform(unified[feat_cols])

joblib.dump(scaler,    SCALER_PATH)
joblib.dump(feat_cols, FEATS_PATH)
unified.to_csv(DATA_PATH, index=False)

print(f'\nUnified     : {len(unified)} rows | {len(feat_cols)} features')
print(f'Features    : {feat_cols}')
print(f'Saved       : {DATA_PATH}')

StudentLife : 46 rows | burnout mean 5.12
Survey      : 262 rows after balancing | low=22 mid=80 high=160

Unified     : 308 rows | 14 features
Features    : ['sleep_hours', 'sleep_quality', 'app_switches_per_hour', 'social_app_ratio', 'productivity_ratio', 'unique_apps_per_day', 'call_count', 'total_call_min', 'missed_call_ratio', 'sms_count', 'sms_sent_ratio', 'screen_time_hours', 'exercise_min_per_week', 'social_hours_per_week']
Saved       : models/unified_burnout_dataset.csv


In [21]:
# Cell 7 — Model + Dataset classes

class BurnoutDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class MLP(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n, 64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32,  1)
        )
    def forward(self, x): return self.net(x).squeeze(-1)

print('Model classes ready')

Model classes ready


In [22]:
# Cell 8 — Train with 5-fold cross-validation

unified   = pd.read_csv(DATA_PATH)
feat_cols = joblib.load(FEATS_PATH)

X = unified[feat_cols].values.astype(np.float32)
y = unified['burnout_score'].values.astype(np.float32)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_maes, fold_r2s = [], []
criterion = nn.HuberLoss(delta=1.0)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    train_loader = DataLoader(BurnoutDS(X_tr, y_tr), batch_size=16, shuffle=True)
    val_loader   = DataLoader(BurnoutDS(X_val, y_val), batch_size=16)

    model = MLP(len(feat_cols)).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=6, factor=0.5)

    best_val, patience_ctr, best_state = np.inf, 0, None
    for epoch in range(1, 301):
        model.train()
        for Xb,yb in train_loader:
            Xb,yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            criterion(model(Xb), yb).backward()
            opt.step()
        model.eval()
        vloss, preds, targets = 0, [], []
        with torch.no_grad():
            for Xb,yb in val_loader:
                out = model(Xb.to(DEVICE))
                vloss += criterion(out, yb.to(DEVICE)).item()
                preds.extend(out.cpu().numpy())
                targets.extend(yb.numpy())
        vloss /= max(len(val_loader),1)
        sched.step(vloss)
        if vloss < best_val - 1e-4:
            best_val, patience_ctr = vloss, 0
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= 20: break

    model.load_state_dict(best_state)
    mae = mean_absolute_error(targets, preds)
    r2  = r2_score(targets, preds)
    fold_maes.append(mae)
    fold_r2s.append(r2)
    print(f'Fold {fold}  MAE={mae:.4f}  R²={r2:.4f}')

print(f'\nMean MAE : {np.mean(fold_maes):.4f} ± {np.std(fold_maes):.4f}')
print(f'Mean R²  : {np.mean(fold_r2s):.4f} ± {np.std(fold_r2s):.4f}')

# retrain on full data for deployment
full_loader = DataLoader(BurnoutDS(X, y), batch_size=16, shuffle=True)
model = MLP(len(feat_cols)).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-3)
for epoch in range(200):
    model.train()
    for Xb,yb in full_loader:
        Xb,yb = Xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        criterion(model(Xb), yb).backward()
        opt.step()

torch.save(model.state_dict(), MODEL_PATH)
print(f'\nFinal model saved → {MODEL_PATH}')

Fold 1  MAE=0.7520  R²=0.7189
Fold 2  MAE=0.9187  R²=0.5122
Fold 3  MAE=0.6822  R²=0.6285
Fold 4  MAE=0.7314  R²=0.6647
Fold 5  MAE=0.7486  R²=0.7412

Mean MAE : 0.7666 ± 0.0801
Mean R²  : 0.6531 ± 0.0809

Final model saved → models/best_burnout_model.pt


In [23]:
# Cell 9 — Quick test (low / moderate / high burnout)

feat_cols = joblib.load(FEATS_PATH)
scaler    = joblib.load(SCALER_PATH)
model     = MLP(len(feat_cols)).to('cpu')
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()

def make_input(overrides):
    defaults = {
        'sleep_hours':6.5, 'sleep_quality':3.0,
        'screen_time_hours':6.0, 'unlock_count':30,
        'avg_session_min':8.0, 'evening_screen_ratio':0.3,
        'app_switches_per_hour':40, 'social_app_ratio':0.2,
        'productivity_ratio':0.3, 'unique_apps_per_day':50,
        'call_count':5, 'total_call_min':10,
        'missed_call_ratio':0.1, 'sms_count':20,
        'sms_sent_ratio':0.4, 'dining_visits':2,
        'social_audio_ratio':0.2, 'active_ratio':0.3,
        'exercise_min_per_week':90, 'social_hours_per_week':5,
        'stationary_ratio':0.5, 'conversation_count':3,
        'total_conv_min':15,
    }
    defaults.update(overrides)
    return np.array([[defaults.get(f, 0.0) for f in feat_cols]], dtype=np.float32)

test_cases = {
    'Low burnout':      {'sleep_hours':8.5,'sleep_quality':5,'screen_time_hours':3,
                         'app_switches_per_hour':20,'exercise_min_per_week':300,
                         'social_hours_per_week':15,'evening_screen_ratio':0.05,
                         'active_ratio':0.6,'social_audio_ratio':0.4,
                         'productivity_ratio':0.6,'social_app_ratio':0.1},
    'Moderate burnout': {'sleep_hours':6.5,'sleep_quality':3,'screen_time_hours':7,
                         'app_switches_per_hour':50,'exercise_min_per_week':60,
                         'social_hours_per_week':5,'evening_screen_ratio':0.3,
                         'active_ratio':0.3,'social_audio_ratio':0.2,
                         'productivity_ratio':0.3,'social_app_ratio':0.3},
    'High burnout':     {'sleep_hours':4.0,'sleep_quality':1,'screen_time_hours':12,
                         'app_switches_per_hour':90,'exercise_min_per_week':20,
                         'social_hours_per_week':1,'evening_screen_ratio':0.6,
                         'active_ratio':0.1,'social_audio_ratio':0.05,
                         'productivity_ratio':0.1,'social_app_ratio':0.6},
}

print(f'Features used: {len(feat_cols)}\n')
for label, vals in test_cases.items():
    x = make_input(vals)
    x = scaler.transform(x)
    with torch.no_grad():
        pred = float(np.clip(model(torch.tensor(x)).item(), 1, 10))
    level = 'Low' if pred<4 else 'Moderate' if pred<7 else 'High'
    print(f'{label:<22} → {round(pred,2):4.1f}  [{level}]') 

Features used: 14

Low burnout            →  6.6  [Moderate]
Moderate burnout       →  6.2  [Moderate]
High burnout           →  1.0  [Low]


In [24]:
# Cell 10 — Gradio demo
import gradio as gr

feat_cols = joblib.load(FEATS_PATH)
scaler    = joblib.load(SCALER_PATH)
model     = MLP(len(feat_cols)).to('cpu')
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()

slider_config = {
    'sleep_hours':           (0, 12,  7.0, 'Sleep hours'),
    'sleep_quality':         (1,  5,  3.0, 'Sleep quality (1–5)'),
    'screen_time_hours':     (0, 16,  6.0, 'Screen time (hours)'),
    'unlock_count':          (0, 150, 30.0,'Phone unlocks per day'),
    'avg_session_min':       (0, 60,  8.0, 'Avg session length (min)'),
    'evening_screen_ratio':  (0,  1,  0.3, 'Evening screen ratio (0–1)'),
    'app_switches_per_hour': (0, 120, 40.0,'App switches per hour'),
    'social_app_ratio':      (0,  1,  0.2, 'Social app ratio (0–1)'),
    'productivity_ratio':    (0,  1,  0.3, 'Productivity app ratio (0–1)'),
    'unique_apps_per_day':   (0, 150, 50.0,'Unique apps per day'),
    'call_count':            (0, 50,  5.0, 'Daily calls'),
    'total_call_min':        (0, 120, 10.0,'Total call minutes'),
    'missed_call_ratio':     (0,  1,  0.1, 'Missed call ratio (0–1)'),
    'sms_count':             (0, 200, 20.0,'Daily SMS count'),
    'sms_sent_ratio':        (0,  1,  0.4, 'SMS sent ratio (0–1)'),
    'dining_visits':         (0, 10,  2.0, 'Dining visits per day'),
    'social_audio_ratio':    (0,  1,  0.2, 'Social audio ratio (0–1)'),
    'active_ratio':          (0,  1,  0.3, 'Physical activity ratio (0–1)'),
    'exercise_min_per_week': (0, 600, 90.0,'Exercise (min/week)'),
    'social_hours_per_week': (0, 40,  5.0, 'Social hours/week'),
}

def predict(*args):
    try:
        x = np.array(args, dtype=np.float32).reshape(1, -1)
        x = scaler.transform(x)
        with torch.no_grad():
            pred = float(np.clip(model(torch.tensor(x)).item(), 1, 10))
        level = ('Low burnout risk', 'You are doing well. Keep up the healthy digital habits.') if pred < 4 \
            else ('Moderate burnout risk', 'Watch your screen time and sleep. Take breaks.') if pred < 7 \
            else ('High burnout risk', 'Seriously consider reducing screen time, improving sleep, and adding exercise.')
        return round(pred, 2), level[0], level[1]
    except Exception as e:
        return 0.0, str(e), ''

inputs = []
for f in feat_cols:
    if f in slider_config:
        mn, mx, default, label = slider_config[f]
        inputs.append(gr.Slider(mn, mx, value=default, step=0.1, label=label))
    else:
        inputs.append(gr.Slider(0, 100, value=0.0, step=0.1, label=f))

gr.Interface(
    fn=predict,
    inputs=inputs,
    outputs=[
        gr.Number(label='Burnout risk score (1–10)'),
        gr.Textbox(label='Risk level'),
        gr.Textbox(label='Recommendation'),
    ],
    title='Digital Burnout Detector',
    description='Adjust your daily digital behavior patterns to see your burnout risk score.',
    examples=[
        [8.5, 5, 3,  20, 5,  0.05, 20, 0.1, 0.6, 30,  5, 15, 0.0, 15, 0.5, 3, 0.3, 0.5, 300, 15],
        [6.5, 3, 7,  50, 8,  0.30, 50, 0.3, 0.2, 60,  8, 20, 0.2, 30, 0.4, 2, 0.2, 0.3,  60,  5],
        [4.0, 1, 12, 90, 3,  0.60, 90, 0.6, 0.1, 100, 20, 60, 0.5, 80, 0.2, 1, 0.1, 0.1,  20,  1],
    ],
    flagging_mode='never',
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [30]:
# Cell — Full evaluation report

from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import matplotlib.pyplot as plt

feat_cols = joblib.load(FEATS_PATH)
scaler    = joblib.load(SCALER_PATH)
model     = MLP(len(feat_cols)).to('cpu')
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()

unified   = pd.read_csv(DATA_PATH)
X = unified[feat_cols].values.astype(np.float32)
y = unified['burnout_score'].values.astype(np.float32)

# predict on full dataset
with torch.no_grad():
    preds = model(torch.tensor(X)).numpy()
preds = np.clip(preds, 1, 10)

mae  = mean_absolute_error(y, preds)
rmse = np.sqrt(mean_squared_error(y, preds))
r2   = r2_score(y, preds)

# burnout level accuracy (Low/Moderate/High)
def to_level(arr):
    return np.where(arr < 4, 0, np.where(arr < 7, 1, 2))

true_levels = to_level(y)
pred_levels = to_level(preds)
level_acc   = (true_levels == pred_levels).mean() * 100

print("── Regression Metrics ───────────────────")
print(f"  MAE        : {mae:.4f}  (avg error on 1–10 scale)")
print(f"  RMSE       : {rmse:.4f}")
print(f"  R²         : {r2:.4f}  (1.0 = perfect)")
print()
print("── Classification Accuracy ──────────────")
print(f"  Level acc  : {level_acc:.1f}%  (Low / Moderate / High)")
print()

# per-level breakdown
from collections import Counter
labels = ['Low (1–4)', 'Moderate (4–7)', 'High (7–10)']
print("── Per-level breakdown ──────────────────")
for i, label in enumerate(labels):
    mask     = true_levels == i
    if mask.sum() == 0: continue
    correct  = (pred_levels[mask] == true_levels[mask]).mean() * 100
    count    = mask.sum()
    print(f"  {label:<18} : {correct:.1f}% correct  ({count} samples)")

print()
print("── Feature importance (correlation) ─────")
unscaled_X = scaler.inverse_transform(X)
corrs = []
for i, f in enumerate(feat_cols):
    corr = np.corrcoef(unscaled_X[:,i], y)[0,1]
    corrs.append((f, abs(corr), corr))
corrs.sort(key=lambda x: x[1], reverse=True)
for f, abs_c, c in corrs[:10]:
    if np.isnan(abs_c):
        continue
    direction = "↑ burnout" if c > 0 else "↓ burnout"
    bar = "█" * int(abs_c * 20)
    print(f"  {f:<30} {abs_c:.3f}  {direction}  {bar}")

── Regression Metrics ───────────────────
  MAE        : 0.5404  (avg error on 1–10 scale)
  RMSE       : 0.7336
  R²         : 0.8482  (1.0 = perfect)

── Classification Accuracy ──────────────
  Level acc  : 86.7%  (Low / Moderate / High)

── Per-level breakdown ──────────────────
  Low (1–4)          : 53.1% correct  (32 samples)
  Moderate (4–7)     : 85.0% correct  (113 samples)
  High (7–10)        : 94.5% correct  (163 samples)

── Feature importance (correlation) ─────
  sleep_quality                  0.615  ↓ burnout  ████████████
  screen_time_hours              0.594  ↑ burnout  ███████████
  sms_sent_ratio                 0.181  ↓ burnout  ███
  missed_call_ratio              0.155  ↓ burnout  ███
  social_app_ratio               0.153  ↓ burnout  ███
  sms_count                      0.151  ↓ burnout  ███
  exercise_min_per_week          0.146  ↓ burnout  ██
  app_switches_per_hour          0.130  ↑ burnout  ██
  productivity_ratio             0.129  ↓ burnout  ██


In [28]:
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import matplotlib.pyplot as plt

In [31]:
# Cell — Download model files
from IPython.display import FileLink
import shutil, os

# zip all model files together
shutil.make_archive('burnout_models_final_final', 'zip', 'models')
print('Created burnout_models.zip')
print('Files inside:')
for f in os.listdir('models'):
    print(f'  {f}')

Created burnout_models.zip
Files inside:
  burnout_scaler.pkl
  unified_burnout_dataset.csv
  feature_cols.pkl
  best_burnout_model.pt
